In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_silver_Group"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Bronze/Groups_Inventory" # ← Change source path
TARGET_PATH = "abfss://silver/Dim_Group" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_silver_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_silver_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_silver_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, 1ac00c88-5b71-43ab-9399-a685e51938a5, 3, Finished, Available, Finished)

🔧 Initializing ntk_silver_Group...
🚀 Starting ntk_silver_Group


In [2]:
source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Bronze_lakehouse.Lakehouse/Files/Bronze_layer/SharePointFiles"
target_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting"

StatementMeta(, 1ac00c88-5b71-43ab-9399-a685e51938a5, 4, Finished, Available, Finished)

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lit, current_timestamp, to_timestamp, row_number, when,
    upper, regexp_replace, desc
)
from pyspark.sql.types import StringType, IntegerType, DoubleType
from pyspark.sql.window import Window
from datetime import datetime

# Step 1: Initialize Spark
spark = SparkSession.builder.appName("BronzeToSilver_GroupInventory").getOrCreate()

today = datetime.now()  
from datetime import datetime, timedelta
today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d") 
bronze_path = f"{source_path}/{year}/{month}/{day}/Groups_Inventory.csv"

# Step 2: Date-based path setup
current_date = datetime.now()
year = str(current_date.year)
month = f"{current_date.month:02d}"
day = f"{current_date.day:02d}"
silver_path = f"{target_path}/{year}/{month}/{day}/Dim_Group.parquet"

# Step 3: Read CSV
df_raw = spark.read.option("header", True).csv(bronze_path)
df_raw.show(1)

StatementMeta(, 1ac00c88-5b71-43ab-9399-a685e51938a5, 5, Finished, Available, Finished)

+-------+---------+--------------------+---------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+----+-----------+---------------+---------------+------------------+---------------+-----------+----------+----------------+--------------+--------------+-----------------------------+-------------+------------+------------------------------+--------------------------+-----------------------+----------------------------+------------------------------+-----------------+-----------+--------------+--------------------+----------+-----------------+----------+--------------------+-----------------------+--------------------+----------------+--------------+---------------+--------------------+------------+
|GroupId|GroupGuid|           LoginName|  PrincipalType|              SiteId|     SiteName|             SiteUrl|               WebId|               Title|         Description| GroupType

In [4]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# Step 4: Add metadata
df_raw = df_raw.withColumn("SnapshotDate", current_timestamp()).withColumn("ProcessedDate", current_timestamp()).withColumn("DataSource", lit("SharePoint_Group_Inventory"))

# Step 5: Convert all columns to string
for field in df_raw.schema.fields:
    if field.name not in ["SnapshotDate", "ProcessedDate"]:
        df_raw = df_raw.withColumn(field.name, col(field.name).cast(StringType()))

# Trim string columns
string_cols = [f.name for f in df_raw.schema.fields if f.dataType == StringType() and f.name not in ["SnapshotDate", "ProcessedDate", "DataSource"]]
for c in string_cols:
    df_raw = df_raw.withColumn(c, trim(col(c)))

# Replace null representations
null_replacements = ["", "NULL", "null", "N/A", "n/a"]
df_cleaned = df_raw
for val in null_replacements:
    df_cleaned = df_cleaned.replace(val, None)

# Step 6: Boolean columns
boolean_columns = [
    "IsSystemGroup","IsHiddenInUI","OnlyAllowMembersViewMembership",
    "AllowMembersEditMembership","AllowRequestToJoinLeave","AutoAcceptRequestToJoinLeave",
    "ReviewRequired","SafeModeUsed","HasDirectPermissions"
]
actual_boolean_columns = [c for c in boolean_columns if c in df_cleaned.columns]

for bool_col in actual_boolean_columns:
    df_cleaned = df_cleaned.withColumn(
        f"{bool_col}_Clean",
        when(upper(col(bool_col)).isin(["TRUE","1","YES","Y"]), True)
        .when(upper(col(bool_col)).isin(["FALSE","0","NO","N"]), False)
        .otherwise(None)
    )

# Step 7: Date columns
date_columns = ["ExpirationDate","CreatedDate","ModifiedDate","RecordedDateTime"]
actual_date_columns = [c for c in date_columns if c in df_cleaned.columns]

for date_col in actual_date_columns:
    df_cleaned = df_cleaned.withColumn(
        f"{date_col}_Clean",
        when(col(date_col).isNotNull() & (col(date_col) != ""), to_timestamp(col(date_col), "M/d/yyyy H:mm"))
        .otherwise(None)
    )
# Step 8: Numeric columns
numeric_columns = ["GroupGuid","GroupId","MemberCount","OwnerCount","AssignedPermissionCount"]
actual_numeric_columns = [c for c in numeric_columns if c in df_cleaned.columns]

for num_col in actual_numeric_columns:
    df_cleaned = df_cleaned.withColumn(
        f"{num_col}_Clean",
        when(col(num_col).rlike("^[0-9]+$"), col(num_col).cast(IntegerType()))
        .when(col(num_col).rlike("^[0-9]*\\.?[0-9]+[Ee][+-]?[0-9]+$"), col(num_col).cast(DoubleType()).cast(IntegerType()))
        .otherwise(None)
    )
# Step 9: Deduplication
dedup_cols = []
if "RecordedDateTime_Clean" in df_cleaned.columns:
    dedup_cols.append(desc("RecordedDateTime_Clean"))
elif "GroupGuid_Clean" in df_cleaned.columns:
    dedup_cols.append(desc("GroupGuid_Clean"))
else:
    dedup_cols.append(desc("GroupGuid"))

window_spec = Window.partitionBy("GroupGuid_Clean").orderBy(*dedup_cols)
df_dedup = df_cleaned.withColumn("row_num", row_number().over(window_spec)) \
                     .filter(col("row_num") == 1).drop("row_num")

# -----------------------------
# Step 10: Final column mapping
# -----------------------------
column_mapping = [
    "GroupGuid","LoginName","GroupId","PrincipalType","Scope","OwnerLoginName","OwnerEmail",
    "MemberCount","ExpirationDate","CreatedBy","CreatedDate","LastModifiedBy","ModifiedDate","GroupType",
    "IsSystemGroup","IsHiddenInUI","OnlyAllowMembersViewMembership",
    "AllowMembersEditMembership","AllowRequestToJoinLeave","AutoAcceptRequestToJoinLeave",
    "RequestToJoinLeaveEmailSetting","OwnerTitle","OwnerCount","AssignedPermissions",
    "AssignedPermissionCount","HasDirectPermissions","ComplianceStatus","ReviewRequired",
    "RecordedDateTime","SafeModeUsed","SiteId","SiteName","SiteUrl","WebId","Title","Description"
]

# Map clean columns to final columns if they exist
final_columns = []
for c in column_mapping:
    clean_col = f"{c}_Clean"
    if clean_col in df_dedup.columns:
        final_columns.append(col(clean_col).alias(c))
    elif c in df_dedup.columns:
        final_columns.append(col(c))


df_final = df_dedup.select(*final_columns)
df_final = df_final.withColumnRenamed("RecordedDateTime", "SnapshotDate")
df_final.show(1)

StatementMeta(, 1ac00c88-5b71-43ab-9399-a685e51938a5, 6, Finished, Available, Finished)

+---------+--------------------+-------+---------------+--------------------+----------+-----------+----------+-------------+------------+------------------------------+--------------------------+-----------------------+----------------------------+------------------------------+----------+----------+--------------------+-----------------------+--------------------+----------------+--------------+-------------------+------------+--------------------+-------------+--------------------+--------------------+--------------------+-----------+
|GroupGuid|           LoginName|GroupId|  PrincipalType|      OwnerLoginName|OwnerEmail|MemberCount| GroupType|IsSystemGroup|IsHiddenInUI|OnlyAllowMembersViewMembership|AllowMembersEditMembership|AllowRequestToJoinLeave|AutoAcceptRequestToJoinLeave|RequestToJoinLeaveEmailSetting|OwnerTitle|OwnerCount| AssignedPermissions|AssignedPermissionCount|HasDirectPermissions|ComplianceStatus|ReviewRequired|       SnapshotDate|SafeModeUsed|              SiteId|  

In [5]:
from pyspark.sql.functions import sha2, col

# Use SHA256 hash (more secure)
df_final = df_final.withColumn("GroupKey", sha2(col("LoginName"), 256))

# Get all columns except GroupKey, then put GroupKey first
other_columns = [col for col in df_final.columns if col != "GroupKey"]
df_final = df_final.select("GroupKey", *other_columns)

# df_final.show(5)
# df_final.printSchema()


StatementMeta(, 1ac00c88-5b71-43ab-9399-a685e51938a5, 7, Finished, Available, Finished)

In [6]:
print(f"File:{silver_path}")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# -----------------------------
# Step 11: Write to Silver
# -----------------------------
try:
    df_final.write.mode("overwrite").option("compression", "snappy").parquet(silver_path)
    print(f"Successfully wrote Group_inventory data to Silver layer: {silver_path}")
except Exception as e:
    print(f"Error writing to Silver layer: {str(e)}")
    raise

StatementMeta(, 1ac00c88-5b71-43ab-9399-a685e51938a5, 8, Finished, Available, Finished)

File:abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/Dim_Group.parquet
Successfully wrote Group_inventory data to Silver layer: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/Dim_Group.parquet


In [7]:
from datetime import datetime

# Define the log_etl_activity function for logging ETL process
def log_etl_activity(status, start_time, rows_read=None, rows_written=None, bytes_processed=None, error_details=None):

    end_time = datetime.now()
    duration_seconds = (end_time - start_time).total_seconds()

    log_message = {
        'Status': status,
        'StartTime': start_time,
        'EndTime': end_time,
        'DurationSeconds': duration_seconds,
        'RowsRead': rows_read,
        'RowsWritten': rows_written,
        'BytesProcessed': bytes_processed,
        'ErrorDetails': error_details
    }

    # For simplicity, let's print the log message (this can be replaced with a logging system)
    print("Logging ETL Activity:", log_message)

# Ensure processing_successful is defined before this block
try:
    NOTEBOOK_NAME = "ETL_Pipeline_Example"  # Define your notebook name or use the existing one
    start_time = datetime.now()  # Capture the start time of the ETL process

    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Simulated metrics
    rows_read = df_raw.count()   # Correct this to have a meaningful `rows_read`
    rows_written = df_cleaned.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details=error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, 1ac00c88-5b71-43ab-9399-a685e51938a5, 9, Finished, Available, Finished)

🔄 Starting ETL processing for ETL_Pipeline_Example...
Logging ETL Activity: {'Status': 'SUCCESS', 'StartTime': datetime.datetime(2025, 10, 15, 5, 13, 27, 816639), 'EndTime': datetime.datetime(2025, 10, 15, 5, 13, 28, 284891), 'DurationSeconds': 0.468252, 'RowsRead': 212, 'RowsWritten': 212, 'BytesProcessed': 524288000, 'ErrorDetails': None}
🎉 ETL_Pipeline_Example pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 212 → 212
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ETL_Pipeline_Example:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ETL_Pipeline_Example logging completed!
